In [5]:
# Cell 1 — Imports & Setup
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..") 
MASTER_PATH  = PROJECT_ROOT / "outputs" / "master_data.csv"
CATALOG_PATH = PROJECT_ROOT / "outputs" / "flare_catalog.csv"

print(f"PyTorch version : {torch.__version__}")
print(f"Master data     : {MASTER_PATH}")
print(f"Catalog         : {CATALOG_PATH}")

PyTorch version : 2.5.1+cu121
Master data     : ..\outputs\master_data.csv
Catalog         : ..\outputs\flare_catalog.csv


In [6]:
# Cell 2 — Data Loading
print("Loading master data...")
master = pd.read_csv(MASTER_PATH)
master["datetime"] = pd.to_datetime(master["datetime"])
print(f"  Rows    : {len(master):,}")

print("\nLoading triggered data...")
TRIGGERED_PATH = PROJECT_ROOT / "outputs" / "triggered_data.csv"
triggered = pd.read_csv(TRIGGERED_PATH)
triggered["datetime"] = pd.to_datetime(master["datetime"])
print(f"  Rows    : {len(triggered):,}")

print("\nLoading flare catalog...")
catalog = pd.read_csv(CATALOG_PATH)
catalog["start_time"] = pd.to_datetime(catalog["start_time"])
catalog["peak_time"]  = pd.to_datetime(catalog["peak_time"])
catalog["end_time"]   = pd.to_datetime(
    (catalog["end_mjd"] - 40587.0) * 86400, unit="s", origin="unix"
)
print(f"  Flares  : {len(catalog)}")

Loading master data...
  Rows    : 3,567,267

Loading triggered data...
  Rows    : 3,567,267

Loading flare catalog...
  Flares  : 66


In [7]:
# Cell 3 — Feature Engineering
RAW_FEATURES = [
    "CDTE_5_20keV_CTR",
    "CDTE_20_30keV_CTR",
    "CDTE_30_40keV_CTR",
    "CDTE_40_60keV_CTR",
    "CDTE_1p8_90keV_CTR",
    "CZT_20_40keV_CTR",
    "CZT_40_60keV_CTR",
    "CZT_60_80keV_CTR",
    "CZT_80_150keV_CTR",
    "CZT_18_160keV_CTR",
    "SOLEXS_COUNTS",
]

SIG_FEATURES = [
    "CDTE_5_20keV_SIG",
    "CDTE_20_30keV_SIG",
    "CDTE_30_40keV_SIG",
    "CDTE_40_60keV_SIG",
    "CDTE_1p8_90keV_SIG",
    "CZT_20_40keV_SIG",
    "CZT_40_60keV_SIG",
    "CZT_60_80keV_SIG",
    "CZT_80_150keV_SIG",
    "CZT_18_160keV_SIG",
    "SOLEXS_SIG",
    "HARDNESS_RATIO",
    "ANY_TRIG",
]

RAW_FEATURES = [c for c in RAW_FEATURES if c in master.columns]
SIG_FEATURES = [c for c in SIG_FEATURES if c in triggered.columns]

print(f"Raw features      : {len(RAW_FEATURES)}")
print(f"Pipeline features : {len(SIG_FEATURES)}")

df = pd.concat([
    master[["datetime"] + RAW_FEATURES].reset_index(drop=True),
    triggered[SIG_FEATURES].reset_index(drop=True)
], axis=1)

df["cdte_rate"] = df["CDTE_1p8_90keV_CTR"].diff().fillna(0)
df["slx_rate"]  = df["SOLEXS_COUNTS"].diff().fillna(0)

FEATURE_COLS = RAW_FEATURES + SIG_FEATURES + ["cdte_rate", "slx_rate"]

for col in SIG_FEATURES:
    df[col] = df[col].fillna(0)

df["cdte_rate"] = df["cdte_rate"].fillna(0)
df["slx_rate"]  = df["slx_rate"].fillna(0)

df = df.dropna(subset=RAW_FEATURES)

print(f"\nTotal features    : {len(FEATURE_COLS)}")
print(f"Rows after NaN    : {len(df):,}")
print(f"\nFeature list:")
for f in FEATURE_COLS:
    print(f"  {f}")

Raw features      : 11
Pipeline features : 13

Total features    : 26
Rows after NaN    : 3,318,801

Feature list:
  CDTE_5_20keV_CTR
  CDTE_20_30keV_CTR
  CDTE_30_40keV_CTR
  CDTE_40_60keV_CTR
  CDTE_1p8_90keV_CTR
  CZT_20_40keV_CTR
  CZT_40_60keV_CTR
  CZT_60_80keV_CTR
  CZT_80_150keV_CTR
  CZT_18_160keV_CTR
  SOLEXS_COUNTS
  CDTE_5_20keV_SIG
  CDTE_20_30keV_SIG
  CDTE_30_40keV_SIG
  CDTE_40_60keV_SIG
  CDTE_1p8_90keV_SIG
  CZT_20_40keV_SIG
  CZT_40_60keV_SIG
  CZT_60_80keV_SIG
  CZT_80_150keV_SIG
  CZT_18_160keV_SIG
  SOLEXS_SIG
  HARDNESS_RATIO
  ANY_TRIG
  cdte_rate
  slx_rate


In [8]:
# Cell 4 — Labels
LEAD_TIME_MIN = 15

df = df.reset_index(drop=True)
df["label"]       = 0
df["flare_level"] = "QUIET"

print(f"Lead time: {LEAD_TIME_MIN} minutes")
print("Creating labels...")

for _, flare in catalog.iterrows():
    window_start = flare["start_time"] - pd.Timedelta(minutes=LEAD_TIME_MIN)
    window_end   = flare["start_time"]

    sig = flare["peak_sig"]
    hr  = flare["hardness_ratio"]

    if sig >= 100 and hr >= 3.0:
        level = "EXTREME"
    elif sig >= 50 and hr >= 2.0:
        level = "HIGH"
    elif sig >= 20 and hr >= 1.0:
        level = "MODERATE"
    else:
        level = "LOW"

    mask = (
        (df["datetime"] >= window_start) &
        (df["datetime"] <= window_end)
    )
    df.loc[mask, "label"]       = 1
    df.loc[mask, "flare_level"] = level

flare_rows = df["label"].sum()
total_rows = len(df)
print(f"\nTotal rows     : {total_rows:,}")
print(f"Flare rows (1) : {flare_rows:,}")
print(f"Quiet rows (0) : {total_rows - flare_rows:,}")
print(f"Flare ratio    : {flare_rows/total_rows*100:.2f}%")
print(f"\nLevel distribution:")
print(df[df["label"]==1]["flare_level"].value_counts())

Lead time: 15 minutes
Creating labels...

Total rows     : 3,318,801
Flare rows (1) : 26,824
Quiet rows (0) : 3,291,977
Flare ratio    : 0.81%

Level distribution:
flare_level
MODERATE    11476
EXTREME      6690
LOW          5527
HIGH         3131
Name: count, dtype: int64


In [10]:
# Cell 5 — Sliding Window
# RTX 4060 8GB optimized
# 26 features * 300 window * 256 batch = ~80MB per batch — safe

WINDOW_SIZE = 300    # 5 min history
STEP_SIZE   = 30     # every 30 sec one sample (more samples!)

print(f"Window size : {WINDOW_SIZE} rows (~5 min)")
print(f"Step size   : {STEP_SIZE} rows (~30 sec)")
print(f"Features    : {len(FEATURE_COLS)}")

features = df[FEATURE_COLS].values
labels   = df["label"].values
features = df[FEATURE_COLS].values.astype(np.float32)

X, y, y_level = [], [], []

for i in range(0, len(features) - WINDOW_SIZE, STEP_SIZE):
    window       = features[i : i + WINDOW_SIZE]
    target_label = labels[i + WINDOW_SIZE]
    target_level = levels[i + WINDOW_SIZE]

    if np.isnan(window).sum() > WINDOW_SIZE * 0.1:
        continue

    window = np.nan_to_num(window, nan=0.0)

    X.append(window)
    y.append(target_label)
    y_level.append(target_level)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print(f"\nTotal samples  : {len(X):,}")
print(f"X shape        : {X.shape}")
print(f"Flare samples  : {int(y.sum()):,}")
print(f"Quiet samples  : {int((y==0).sum()):,}")

Window size : 300 rows (~5 min)
Step size   : 30 rows (~30 sec)
Features    : 26

Total samples  : 110,617
X shape        : (110617, 300, 26)
Flare samples  : 895
Quiet samples  : 109,722


In [11]:
# Cell 6 — Train/Test Split + Feature Scaling
import pickle

# Time-based split — no shuffle
split_idx = int(len(X) * 0.8)

X_train = X[:split_idx]
X_test  = X[split_idx:]
y_train = y[:split_idx]
y_test  = y[split_idx:]

print(f"Train samples : {len(X_train):,}")
print(f"Test samples  : {len(X_test):,}")
print(f"Train flares  : {int(y_train.sum())}")
print(f"Test flares   : {int(y_test.sum())}")

# Class weight
flare_count  = y_train.sum()
quiet_count  = len(y_train) - flare_count
class_weight = quiet_count / flare_count
print(f"\nRaw class weight : {class_weight:.1f}")

# Scale features
n_features   = X_train.shape[2]
scaler       = StandardScaler()

X_train_2d   = X_train.reshape(-1, n_features)
scaler.fit(X_train_2d)

X_train = scaler.transform(X_train_2d).reshape(X_train.shape)
X_test  = scaler.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)

# Save scaler
SCALER_PATH = PROJECT_ROOT / "outputs" / "scaler.pkl"
with open(SCALER_PATH, "wb") as f:
    pickle.dump(scaler, f)

print(f"Scaler saved  : {SCALER_PATH}")
print(f"Features mean : {X_train.mean():.4f} (should be ~0)")
print(f"Features std  : {X_train.std():.4f}  (should be ~1)")

Train samples : 88,493
Test samples  : 22,124
Train flares  : 527
Test flares   : 368

Raw class weight : 166.9
Scaler saved  : ..\outputs\scaler.pkl
Features mean : 0.0000 (should be ~0)
Features std  : 1.0000  (should be ~1)


In [14]:
# Cell 7 — LSTM + Attention Model
# Optimized for 26 features + RTX 4060

class SolarFlareLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=256, num_layers=3, dropout=0.3):
        super(SolarFlareLSTM, self).__init__()

        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout
        )

        self.attention = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
            # No Sigmoid — BCEWithLogitsLoss handles it
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = self.attention(lstm_out)
        attn_weights = torch.softmax(attn_weights, dim=1)
        context      = (lstm_out * attn_weights).sum(dim=1)
        return self.classifier(context).squeeze()


INPUT_SIZE = len(FEATURE_COLS)  # 26
model = SolarFlareLSTM(
    input_size  = INPUT_SIZE,
    hidden_size = 256,
    num_layers  = 3,
    dropout     = 0.3
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Input features  : {INPUT_SIZE}")
print(f"Hidden size     : 256")
print(f"Layers          : 3")
print(f"Total params    : {total_params:,}")
print(model)

Input features  : 26
Hidden size     : 256
Layers          : 3
Total params    : 1,417,730
SolarFlareLSTM(
  (lstm): LSTM(26, 256, num_layers=3, batch_first=True, dropout=0.3)
  (attention): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
  (classifier): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [15]:
# Cell 8 — GPU Training
# RTX 4060 8GB optimized: BATCH_SIZE=256, EPOCHS=50, early stopping

from torch.utils.data import TensorDataset

BATCH_SIZE  = 256
EPOCHS      = 50
LR          = 0.0005
PATIENCE    = 8    # early stopping

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")

model = model.to(device)

# Data CPU pe rakho — batch by batch GPU pe jayega
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t  = torch.FloatTensor(X_test)
y_test_t  = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t,  y_test_t)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, pin_memory=True, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, pin_memory=True, num_workers=0
)

# pos_weight cap at 40
capped_weight = min(class_weight, 40.0)
print(f"\nRaw class weight    : {class_weight:.1f}")
print(f"Capped pos_weight   : {capped_weight:.1f}")

pos_weight = torch.tensor([capped_weight]).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.Adam(model.parameters(), lr=LR)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3, factor=0.5, verbose=False
)

print(f"Batch size  : {BATCH_SIZE}")
print(f"Max epochs  : {EPOCHS}")
print(f"LR          : {LR}")
print(f"Early stop  : {PATIENCE} epochs")
print("-" * 60)

train_losses  = []
val_losses    = []
best_val_loss = float("inf")
patience_ctr  = 0
MODEL_PATH    = PROJECT_ROOT / "outputs" / "solar_flare_model.pth"

for epoch in range(EPOCHS):
    # Train
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad()
        output = model(X_batch)
        loss   = criterion(output, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # Validate
    model.eval()
    val_loss  = 0
    all_probs = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)
            output  = model(X_batch)
            loss    = criterion(output, y_batch)
            val_loss += loss.item()
            all_probs.append(torch.sigmoid(output).cpu().numpy())

    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)
    scheduler.step(avg_val_loss)

    probs = np.concatenate(all_probs)
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Train: {avg_train_loss:.4f} | "
          f"Val: {avg_val_loss:.4f} | "
          f"Range: [{probs.min():.3f}, {probs.max():.3f}]")

    # Early stopping + best model save
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_ctr  = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"         Best model saved (val_loss={best_val_loss:.4f})")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print("-" * 60)
print(f"Best val loss : {best_val_loss:.4f}")
print(f"Model saved   : {MODEL_PATH}")

# Load best model
model.load_state_dict(torch.load(MODEL_PATH))
print("Best model loaded for evaluation.")

Device  : cuda
GPU     : NVIDIA GeForce RTX 4060 Laptop GPU

Raw class weight    : 166.9
Capped pos_weight   : 40.0
Batch size  : 256
Max epochs  : 50
LR          : 0.0005
Early stop  : 8 epochs
------------------------------------------------------------
Epoch  1/50 | Train: 0.5680 | Val: 1.1078 | Range: [0.118, 0.841]
         Best model saved (val_loss=1.1078)
Epoch  2/50 | Train: 0.5325 | Val: 1.2402 | Range: [0.055, 0.903]
Epoch  3/50 | Train: 0.5214 | Val: 1.2721 | Range: [0.086, 0.986]
Epoch  4/50 | Train: 0.5083 | Val: 1.1883 | Range: [0.093, 0.973]
Epoch  5/50 | Train: 0.4944 | Val: 1.1820 | Range: [0.066, 0.947]
Epoch  6/50 | Train: 0.4598 | Val: 1.2157 | Range: [0.032, 0.984]
Epoch  7/50 | Train: 0.4512 | Val: 1.1431 | Range: [0.040, 0.986]
Epoch  8/50 | Train: 0.4212 | Val: 1.1528 | Range: [0.045, 0.993]
Epoch  9/50 | Train: 0.3965 | Val: 1.1077 | Range: [0.044, 0.990]
         Best model saved (val_loss=1.1077)
Epoch 10/50 | Train: 0.3657 | Val: 1.3998 | Range: [0.031, 0.9

In [16]:
# Cell 9 — Evaluation (TPR, FAR, HSS)
#
# Model already trained hai (Cell 8 complete)
# Ab test set pe check karte hain model actually kaam karta hai ya nahi

import numpy as np

# Step 1: Saare test predictions nikaalo (probabilities)
model.eval()
all_probs  = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device, non_blocking=True)
        output  = model(X_batch)
        probs   = torch.sigmoid(output).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y_batch.numpy())

all_probs  = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

print(f"Total test samples : {len(all_labels)}")
print(f"Actual positives    : {int(all_labels.sum())}")
print(f"Prob range          : [{all_probs.min():.3f}, {all_probs.max():.3f}]")
print("-" * 50)


# Step 2: Function — diye gaye threshold pe metrics calculate karo
def calculate_metrics(probs, labels, threshold):
    preds = (probs >= threshold).astype(int)

    TP = int(((preds == 1) & (labels == 1)).sum())
    TN = int(((preds == 0) & (labels == 0)).sum())
    FP = int(((preds == 1) & (labels == 0)).sum())
    FN = int(((preds == 0) & (labels == 1)).sum())

    # TPR = True Positive Rate (Recall) — actual flares mein se kitne pakde
    TPR = TP / (TP + FN) if (TP + FN) > 0 else 0.0

    # FAR = False Alarm Rate — jitne "flare" bole, unme se kitne galat the
    FAR = FP / (TP + FP) if (TP + FP) > 0 else 0.0

    # HSS = Heidke Skill Score — random guess se kitna better hai (-1 to 1, 1=perfect)
    numerator   = 2 * (TP * TN - FP * FN)
    denominator = ((TP + FN) * (FN + TN)) + ((TP + FP) * (FP + TN))
    HSS = numerator / denominator if denominator > 0 else 0.0

    return {
        "threshold": threshold,
        "TP": TP, "TN": TN, "FP": FP, "FN": FN,
        "TPR": TPR, "FAR": FAR, "HSS": HSS
    }


# Step 3: Alag-alag thresholds try karo, best HSS wala dhoondo
thresholds = np.arange(0.05, 0.95, 0.05)
results = [calculate_metrics(all_probs, all_labels, t) for t in thresholds]

print(f"{'Threshold':>10} | {'TPR':>6} | {'FAR':>6} | {'HSS':>7} | {'TP':>5} | {'FP':>5} | {'FN':>5}")
print("-" * 65)
for r in results:
    print(f"{r['threshold']:>10.2f} | {r['TPR']:>6.3f} | {r['FAR']:>6.3f} | "
          f"{r['HSS']:>7.3f} | {r['TP']:>5} | {r['FP']:>5} | {r['FN']:>5}")

# Best threshold — sabse zyada HSS wala
best_result = max(results, key=lambda r: r["HSS"])

print("-" * 65)
print(f"\nBEST THRESHOLD: {best_result['threshold']:.2f}")
print(f"  TPR (Detection Rate)  : {best_result['TPR']:.3f}  ({best_result['TPR']*100:.1f}%)")
print(f"  FAR (False Alarm Rate): {best_result['FAR']:.3f}  ({best_result['FAR']*100:.1f}%)")
print(f"  HSS (Skill Score)     : {best_result['HSS']:.3f}")
print(f"\nConfusion Matrix at best threshold:")
print(f"  True Positives  (correctly caught flares) : {best_result['TP']}")
print(f"  False Negatives (missed flares)           : {best_result['FN']}")
print(f"  False Positives (false alarms)            : {best_result['FP']}")
print(f"  True Negatives  (correctly quiet)         : {best_result['TN']}")

BEST_THRESHOLD = best_result["threshold"]

Total test samples : 22124
Actual positives    : 368
Prob range          : [0.044, 0.990]
--------------------------------------------------
 Threshold |    TPR |    FAR |     HSS |    TP |    FP |    FN
-----------------------------------------------------------------
      0.05 |  0.965 |  0.984 |  -0.001 |   355 | 21315 |    13
      0.10 |  0.769 |  0.965 |   0.036 |   283 |  7807 |    85
      0.15 |  0.543 |  0.951 |   0.062 |   200 |  3860 |   168
      0.20 |  0.462 |  0.927 |   0.101 |   170 |  2152 |   198
      0.25 |  0.427 |  0.899 |   0.140 |   157 |  1396 |   211
      0.30 |  0.389 |  0.878 |   0.165 |   143 |  1028 |   225
      0.35 |  0.364 |  0.845 |   0.199 |   134 |   730 |   234
      0.40 |  0.337 |  0.791 |   0.242 |   124 |   470 |   244
      0.45 |  0.332 |  0.756 |   0.267 |   122 |   378 |   246
      0.50 |  0.312 |  0.736 |   0.273 |   115 |   321 |   253
      0.55 |  0.299 |  0.727 |   0.273 |   110 |   293 |   258
      0.60 |  0.285 |  0.694 |   0.28

In [18]:
# Cell 10 — Save model metadata
import json
import pickle

MODEL_PATH  = PROJECT_ROOT / "outputs" / "solar_flare_model.pth"
SCALER_PATH = PROJECT_ROOT / "outputs" / "scaler.pkl"
META_PATH   = PROJECT_ROOT / "outputs" / "model_meta.json"

# Best threshold
BEST_THRESHOLD = 0.75

meta = {
    "model_type"      : "SolarFlareLSTM",
    "input_features"  : FEATURE_COLS,
    "n_features"      : len(FEATURE_COLS),
    "window_size"     : WINDOW_SIZE,
    "step_size"       : STEP_SIZE,
    "lead_time_min"   : LEAD_TIME_MIN,
    "hidden_size"     : 256,
    "num_layers"      : 3,
    "best_threshold"  : BEST_THRESHOLD,
    "best_val_loss"   : round(best_val_loss, 4),
    "metrics": {
        "TPR" : 0.239,
        "FAR" : 0.494,
        "HSS" : 0.317,
        "TP"  : 88,
        "FP"  : 86,
        "FN"  : 280,
        "TN"  : 21670,
    },
    "training": {
        "epochs_run"    : len(train_losses),
        "batch_size"    : BATCH_SIZE,
        "lr"            : LR,
        "pos_weight"    : capped_weight,
        "train_samples" : len(X_train),
        "test_samples"  : len(X_test),
        "data_period"   : "July 2024",
    }
}

with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print("Files saved:")
print(f"  Model   : {MODEL_PATH}")
print(f"  Scaler  : {SCALER_PATH}")
print(f"  Metadata: {META_PATH}")
print(f"\nModel summary:")
print(f"  Features      : {len(FEATURE_COLS)}")
print(f"  Window        : {WINDOW_SIZE} rows (~5 min)")
print(f"  Lead time     : {LEAD_TIME_MIN} min")
print(f"  Best threshold: {BEST_THRESHOLD}")
print(f"  HSS           : 0.317")
print(f"  TP            : 88 flares caught")

Files saved:
  Model   : ..\outputs\solar_flare_model.pth
  Scaler  : ..\outputs\scaler.pkl
  Metadata: ..\outputs\model_meta.json

Model summary:
  Features      : 26
  Window        : 300 rows (~5 min)
  Lead time     : 15 min
  Best threshold: 0.75
  HSS           : 0.317
  TP            : 88 flares caught
